# CUADERNO JUPYTER UNIFICADO: Práctica 1 – Metaheurísticas (Curso 2025/2026)
## HILL CLIMBING (HC)

## 1. Importación de librerías

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

import random
import time
import os

## 2. Carga de series temporales

In [ ]:
def read_serie(path):
    with open(path, "r") as f:
        contenido = f.read().replace("[", "").replace("]", "").split()
    return [float(x) for x in contenido]

files = ["../datos/TS1", "../datos/TS2", "../datos/TS3", "../datos/TS4"]
series = [read_serie(f) for f in files]
k_values = [9, 10, 20, 50]

print("Series cargadas correctamente")

✓ Series cargadas correctamente


## 3. Regresión lineal por segmentos

In [4]:
def estimate_segment_coef(x, y):
    x = np.array(x).reshape(-1, 1)
    y = np.array(y)
    if len(x) < 2:
        return (0.0, 0.0)
    model = LinearRegression().fit(x, y)
    return (model.coef_[0], model.intercept_)

def estimate_all_coef(serie, points):
    coef = []
    start = 0
    pts = points.copy()
    pts.append(len(serie))
    for end in pts:
        x = list(range(start, end))
        y = serie[start:end]
        coef.append(estimate_segment_coef(x, y))
        start = end
    return coef

def estimate_all_points(coef, points, n):
    estimated = []
    start = 0
    pts = points.copy()
    pts.append(n)
    for idx, end in enumerate(pts):
        m, b = coef[idx]
        for i in range(start, end):
            estimated.append(m * i + b)
        start = end
    return estimated

## 4. Generación de puntos de corte aleatorios

In [5]:
def randomPoints(longitud_serie, n_cortes):
    b = []
    while n_cortes > 0:
        r = random.randint(1, longitud_serie - 1)
        if r not in b:
            b.append(r)
            n_cortes -= 1
    b.sort()
    return b

## 5. Cálculo del RMSE por segmentos

In [6]:
def RMSE(y_real, y_pred):
    mse = mean_squared_error(y_real, y_pred)
    return np.sqrt(mse)

def mean_rmse(serie, points):
    rmse_acc = 0.0
    start = 0
    coef = estimate_all_coef(serie, points.copy())
    estimated = estimate_all_points(coef, points.copy(), len(serie))
    pts = points.copy()
    pts.append(len(serie))

    for point in pts:
        rmse_acc += RMSE(serie[start:point], estimated[start:point])
        start = point

    return rmse_acc / len(pts)

## 6. Implementación del algoritmo Hill Climbing (HC)

El algoritmo parte de una solución aleatoria y explora su vecindario modificando los puntos de corte.  
Si encuentra una solución mejor, se mueve hacia ella.  
El proceso continúa hasta que no se encuentran mejoras.

### 6.1 Generación del vecindario en Hill Climbing

Para aplicar Hill Climbing necesitamos definir qué consideramos un **vecino** de una solución.

Una solución está formada por un conjunto ordenado de puntos de corte:



\[
\text{solución} = [p_1, p_2, \dots, p_k]
\]



Cada vecino se genera modificando **uno de los puntos de corte en ±1 posición**, siempre respetando:

- Que ningún punto sea 0  
- Que ningún punto sea igual a la longitud total de la serie  
- Que los puntos sigan **ordenados estrictamente**  
- Que no existan **duplicados**  
- Que no aparezcan **segmentos vacíos** (dos puntos iguales)

Esto garantiza que todos los vecinos representan segmentaciones válidas y evita errores en el cálculo del RMSE.


In [7]:
def search_neighbour(serie, n):
    v = []

    for i, point in enumerate(serie):
        serie_aux = serie.copy()
        serie_aux[i] = point+1
        v.append(serie_aux)

        serie_aux = serie.copy()
        serie_aux[i] = point-1
        v.append(serie_aux)

    p = []

    for vect in v:
        if vect[0] == 0:
            continue
        if vect[len(vect)-1]==n:
            continue
        flag = False
        for i in range(len(vect)-1):
            if vect[i] == vect[i+1]:
                flag = True
                break
        if flag:
            continue
        
        p.append(vect)

    return p

## 6.3. Implementación del algoritmo Hill Climbing

In [8]:
def hc(serie, k, max_iters=200):
    startTime = time.time()

    # Solución inicial
    solucion = randomPoints(len(serie), k)
    rmse = mean_rmse(serie, solucion)

    for _ in range(max_iters):
        vecinos = search_neighbour(solucion, len(serie))

        if not vecinos:
            break

        # Buscar el mejor vecino
        mejor_vecino = None
        mejor_rmse = rmse

        for v in vecinos:
            current = mean_rmse(serie, v)
            if current < mejor_rmse:
                mejor_rmse = current
                mejor_vecino = v

        # Si no mejora, detener
        if mejor_vecino is None:
            break

        # Actualizar solución
        solucion = mejor_vecino
        rmse = mejor_rmse

    endTime = time.time()
    return {'rmse': rmse, 'points': solucion}, endTime - startTime

## 7. Ejecución del experimento completo

Se ejecuta Hill Climbing 20 veces por cada serie temporal para obtener:

- RMSE medio  
- Desviación típica  
- Tiempo medio  
- Mejor solución encontrada  

In [9]:
def ejecutar_HC(series, k_values, repeticiones=20):
    resultados = []

    print("\n=== Ejecutando experimento Hill Climbing ===")

    for idx, serie in enumerate(series):
        k = k_values[idx]
        print(f"\nProcesando TS{idx+1} (k={k})...")

        rep_result = []
        for rep in range(repeticiones):
            sol, tiempo = hc(serie, k)
            rep_result.append({
                "rmse": sol["rmse"],
                "points": sol["points"],
                "tiempo": tiempo
            })
            print(f"Repetición {rep+1}/{repeticiones} completada")

        resultados.append(rep_result)

    print("\n=== Experimento finalizado ===")
    return resultados

In [ ]:
resultados_HC = ejecutar_HC(series, k_values, repeticiones=20)


=== Ejecutando experimento Hill Climbing ===

Procesando TS1 (k=9)...
Repetición 1/20 completada
Repetición 2/20 completada
Repetición 3/20 completada
Repetición 4/20 completada
Repetición 5/20 completada
Repetición 6/20 completada
Repetición 7/20 completada
Repetición 8/20 completada
Repetición 9/20 completada
Repetición 10/20 completada
Repetición 11/20 completada
Repetición 12/20 completada
Repetición 13/20 completada
Repetición 14/20 completada
Repetición 15/20 completada
Repetición 16/20 completada
Repetición 17/20 completada
Repetición 18/20 completada
Repetición 19/20 completada
Repetición 20/20 completada

Procesando TS2 (k=10)...
Repetición 1/20 completada
Repetición 2/20 completada
Repetición 3/20 completada
Repetición 4/20 completada
Repetición 5/20 completada
Repetición 6/20 completada
Repetición 7/20 completada
Repetición 8/20 completada
Repetición 9/20 completada
Repetición 10/20 completada
Repetición 11/20 completada
Repetición 12/20 completada
Repetición 13/20 completa

## 8. Visualización de resultados

### 8.1 Evolución del RMSE
Se representa la media del RMSE y la banda de ±1 desviación típica para las 20 ejecuciones.

In [ ]:
def plot_HC_RMSE(series, resultados):
    for idx in range(len(series)):
        rmse_vals = [rep["rmse"] for rep in resultados[idx]]
        ejecuciones = list(range(1, len(rmse_vals) + 1))

        mean_rmse_val = np.mean(rmse_vals)
        std_rmse_val = np.std(rmse_vals)

        plt.figure(figsize=(10,5))
        plt.title(f"Evolución del RMSE - Hill Climbing (TS{idx+1})")

        # Puntos reales
        plt.plot(ejecuciones, rmse_vals, marker="o", color="blue", label="RMSE por ejecución")

        # Media
        plt.hlines(mean_rmse_val, 1, len(rmse_vals), color="green", linewidth=2, label="Media RMSE")

        # Banda de desviación
        plt.fill_between(
            ejecuciones,
            mean_rmse_val - std_rmse_val,
            mean_rmse_val + std_rmse_val,
            color="green",
            alpha=0.2,
            label="±1 desviación"
        )

        plt.xlabel("Ejecución")
        plt.ylabel("RMSE")
        plt.grid(True)
        plt.legend()

        

        plt.show()

In [ ]:
plot_HC_RMSE(series, resultados_HC)

### 8.2 Representación gráfica de la segmentación final

Se muestra la serie real, la serie estimada y los puntos de corte óptimos encontrados por Hill Climbing.

In [ ]:
def plot_final_HC(series, resultados, k_values):
    for idx, serie in enumerate(series):
        best = min(resultados[idx], key=lambda x: x["rmse"])
        points = best["points"]

        coef = estimate_all_coef(serie, points)
        estimada = estimate_all_points(coef, points, len(serie))

        plt.figure(figsize=(12,5))
        plt.plot(serie, label="Serie real", color="blue")
        plt.plot(estimada, label="Serie estimada", color="red")

        for p in points:
            plt.axvline(x=p, linestyle="--", color="black")

        plt.title(f"Solución final HC - TS{idx+1} (k={k_values[idx]})")
        plt.xlabel("Tiempo")
        plt.ylabel("Valor")
        plt.legend()
        plt.grid(True)
        plt.savefig(f"TS{idx+1}_sol.png", dpi=150)
        plt.show()

plot_final_HC(series, resultados_HC, k_values)

## 9. Obtención de métricas
Se calcula:

- RMSE medio  
- Desviación típica  
- Tiempo medio  


In [ ]:
from tabulate import tabulate

# Cálculo de métricas
rmse_medios = [np.mean([rep["rmse"] for rep in resultados_HC[i]]) for i in range(4)]
rmse_desv = [np.std([rep["rmse"] for rep in resultados_HC[i]]) for i in range(4)]
tiempos_medios = [np.mean([rep["tiempo"] for rep in resultados_HC[i]]) for i in range(4)]

# Crear tabla base
tabla = pd.DataFrame({
    "Serie": ["TS1 (k=9)", "TS2 (k=10)", "TS3 (k=20)", "TS4 (k=50)"],
    "RMSE medio": rmse_medios,
    "Desviación típica": rmse_desv,
    "Tiempo medio (s)": tiempos_medios
})

# Formateo profesional (6 decimales)
tabla_formateada = tabla.copy()
tabla_formateada["RMSE medio"] = tabla_formateada["RMSE medio"].map("{:.6f}".format)
tabla_formateada["Desviación típica"] = tabla_formateada["Desviación típica"].map("{:.6f}".format)
tabla_formateada["Tiempo medio (s)"] = tabla_formateada["Tiempo medio (s)"].map("{:.6f}".format)

# Mostrar tabla con estilo fancy_grid
print(tabulate(tabla_formateada, headers="keys", tablefmt="fancy_grid", showindex=False))